In [ ]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0


# PostgreSQL to Databricks Pipeline Generation

This notebook generates optimized PostgreSQL to Databricks ingestion pipelines using Databricks Asset Bundles.

## Process Overview
1. **Load balancing**: Groups tables into gateway + pipeline configurations
2. **YAML generation**: Creates Databricks Asset Bundle YAML files

## Prerequisites
- Upload this repo (or at least `core/`, `utilities.py`, and `postgres/`) to your Databricks workspace
- Ensure you have a CSV file with your source table list
- Have a Databricks Unity Catalog connection configured for PostgreSQL

Important: **UC POSTGRESQL connections don’t accept a `database` option**. Pick the database per table using `source_database` in the CSV.


In [ ]:
%pip install pyyaml


## Configure Parameters

**IMPORTANT**: Modify these parameters for your environment before running!


In [ ]:
from databricks.sdk import WorkspaceClient

from utilities import load_input_csv
from postgres.connector import PostgreSQLConnector

w = WorkspaceClient()

# Automatically get current user email
username = w.current_user.me().user_name

# Automatically get workspace host URL
workspace_host = w.config.host

# Automatically construct output root path for bundle deployment
WORKSPACE_HOST = workspace_host
ROOT_PATH = f"/Users/{username}/.bundle/${{bundle.name}}/${{bundle.target}}"


In [ ]:
INPUT_CSV = "examples/field_engg/pipeline_config.csv"
OUTPUT_DIR = f"/Workspace/Users/{username}/lakehouse-tapworks/postgres/examples/field_engg/deployment"

df = load_input_csv(INPUT_CSV)

# Create connector
connector = PostgreSQLConnector()

# Generate pipelines
result = connector.run_complete_pipeline_generation(
    df=df,
    output_dir=OUTPUT_DIR,
    targets={
        "dev": {
            "workspace_host": WORKSPACE_HOST,
            "root_path": ROOT_PATH,
        }
    },
    max_tables_per_gateway=250,
    max_tables_per_pipeline=250,
)

In [ ]:
# List generated files
print("\nGenerated DAB Project Structure:")
print(f"\n{OUTPUT_DIR}/")

import os

files_to_check = [
    "databricks.yml",
    "resources/gateways.yml",
    "resources/pipelines.yml",
    "resources/jobs.yml",
]

for file in files_to_check:
    full_path = os.path.join(OUTPUT_DIR, file)
    exists = "✓" if os.path.exists(full_path) else "✗"
    print(f"  {exists} {file}")


## Example: Using Custom Default Values and Overrides

You can use `default_values` and `override_input_config` parameters for advanced configuration:

**`default_values` Parameter:**
- Provides custom default values for optional columns
- Merged with built-in defaults (your values take precedence)
- Applied only to missing/empty values in the CSV

**`override_input_config` Parameter:**
- Forces specific column values for ALL rows
- Overwrites any existing values in the CSV
- Useful for environment-specific overrides

**Example Use Cases:**
- Override schedule for testing (e.g., disable auto-runs)
- Force all pipelines to use specific gateway node types
- Force a specific connection name for a dev workspace

In [ ]:
INPUT_CSV = "examples/field_engg/pipeline_config.csv"
OUTPUT_DIR = f"/Workspace/Users/{username}/lakehouse-tapworks/postgres/examples/field_engg/deployment_overrides"

df = load_input_csv(INPUT_CSV)

connector = PostgreSQLConnector()

result = connector.run_complete_pipeline_generation(
    df=df,
    output_dir=OUTPUT_DIR,
    targets={
        "dev": {
            "workspace_host": WORKSPACE_HOST,
            "root_path": ROOT_PATH,
        }
    },
    max_tables_per_gateway=250,
    max_tables_per_pipeline=250,
    override_input_config={
        "schedule": "0 0 * * 0",
        "gateway_worker_type": "m6d.xlarge",
        "gateway_driver_type": "m6d.xlarge",
    },
)

In [ ]:
INPUT_CSV = "examples/field_engg/pipeline_config.csv"
OUTPUT_DIR = f"/Workspace/Users/{username}/lakehouse-tapworks/postgres/examples/field_engg/deployment_defaults"

df = load_input_csv(INPUT_CSV)

connector = PostgreSQLConnector()

result = connector.run_complete_pipeline_generation(
    df=df,
    output_dir=OUTPUT_DIR,
    targets={
        "dev": {
            "workspace_host": WORKSPACE_HOST,
            "root_path": ROOT_PATH,
        }
    },
    # These defaults apply only where CSV values are missing/empty
    default_values={
        "schedule": "0 */12 * * *",
        "gateway_worker_type": "m7d.xlarge",
        "gateway_driver_type": "m7d.xlarge",
    },
    max_tables_per_gateway=250,
    max_tables_per_pipeline=250,
)